# Task 5: Employee Count per Department

## Step 1: Initialize & Load Clean Data

In [1]:
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

# Load and parse valid records
raw_rdd = sc.textFile("/home/jovyan/data/raw/employees.txt")
header = raw_rdd.first()

def extract_dept(line):
    parts = line.split(",")
    if len(parts) == 9:
        try:
            float(parts[4])
            return [parts[2]]
        except ValueError:
            return []
    return []

dept_rdd = raw_rdd.filter(lambda x: x != header) \
                  .filter(lambda x: x.strip() != "") \
                  .flatMap(extract_dept)

print(f"Departments loaded: {dept_rdd.count()}")

Departments loaded: 9


## Step 2: Count Employees per Department

Map each department to `(dept, 1)` then `reduceByKey`.

In [2]:
dept_counts = dept_rdd.map(lambda x: (x, 1)) \
                      .reduceByKey(lambda a, b: a + b)

print("Employee count per department:\n")
for dept, count in dept_counts.collect():
    print(f"  {dept:<12} → {count}")

Employee count per department:

  Engineering  → 3
  Sales        → 2
  Finance      → 1
  IT           → 1
  HR           → 1
  Marketing    → 1


## Step 3: Sort by Count Descending

In [3]:
sorted_counts = dept_counts.map(lambda x: (x[1], x[0])) \
                           .sortByKey(ascending=False) \
                           .map(lambda x: (x[1], x[0]))

total = sorted_counts.map(lambda x: x[1]).sum()

print(f"{'Department':<14} {'Count':>6}  Visualization")
print("-" * 42)
for dept, count in sorted_counts.collect():
    bar = "█" * count
    pct = count / total * 100
    print(f"  {dept:<12} {count:>4}     {bar} ({pct:.0f}%)")

print("-" * 42)
print(f"  {'Total':<12} {total:>4}")

Department      Count  Visualization
------------------------------------------
  Engineering     3     ███ (33%)
  Sales           2     ██ (22%)
  Finance         1     █ (11%)
  IT              1     █ (11%)
  HR              1     █ (11%)
  Marketing       1     █ (11%)
------------------------------------------
  Total           9


## Step 4: Alternative — Using `countByKey`
A simpler built-in method for this exact use case.

In [ ]:
# Alternative one-liner
alt_counts = dept_rdd.countByValue()

print("Using countByValue (dictionary):\n")
for dept, count in sorted(alt_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {dept:<12} → {count}")

Using countByValue (dictionary):

  Engineering  → 3
  Sales        → 2
  Marketing    → 1
  Finance      → 1
  IT           → 1
  HR           → 1


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 48378)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/local/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/local/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/local/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates =

##  Task 5 Complete